## Middlewares

Middlewares provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
1. Tracking agent behaviour with logging, analytics and debugging.
2. Transforminh prompts, tool selection and output formatting
3. Adding retries, fallbacks and early termination logic
4. Applying rate limits, guardrails, and PII detection

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI-API')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ-API')

# model = init_chat_model("groq:qwen/qwen3.6-27b")
model = init_chat_model("google_genai:gemini-3.5-flash-lite")

## Summarization Middlewares
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
1. Long running conversations that exceed contest windows.
2. Multi turn dialogues with extensive history.
3. Applications where preserving full conversation context matters

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model = model,
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [6]:
### Run with a thread
config = {"configurable" : {"thread_id" : "test-1"}}

questions = [
    "What is 2+2 ?", "What is 10*5 ?", "What is 100/4 ?",
    "What is 15-7 ?", "What is 3*4 ?", "What is 4*4 ?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f'Messages: {response}')

    print(f'Messages: {len(response['messages'])}')


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Messages: {'messages': [HumanMessage(content='What is 2+2 ?', additional_kwargs={}, response_metadata={}, id='5d4dcf7e-f60b-41b4-855f-834a173995ff'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EmAKXgFpFH0T6jA1A0KQjYmfIsUYdVq+LoohOO/1XpvCrLwzbijHihXUYkX9VPAjboQgTn3dfavmRhrpErmYkpKhnWDlZexxsVCuyMOoCZDjkXJ2EOool0CxqqTsqLATIRQ='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f364-9f85-7d22-a4d2-1963e89c3707-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2 ?', additional_kwargs={}, response_metadata={}, id='5d4dcf7e-f60b-41b4-855f-834a173995ff'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EmAKXgFpFH0T6

## Token size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - return long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $100/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens", 550),
            keep = ("tokens", 200),
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

## Token counter approx
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [9]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {'messages' : [HumanMessage(content = f'Find hotels in {city}')]},
        config = config
    )
    print(response)
    tokens = count_token(response['messages'])
    print(f'{city} : ~{tokens}, {len(response['messages'])} messages')
    print(f'{(response['messages'])}')


{'messages': [HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='86d9f7ea-9517-4f4e-a678-81cb0bbbccb0'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_7684140': 'El4KXAERTTIPMKd4HMD+jVa900gbPcorS+MZHwxQ2YBK2tSTO9MMk6dSZ+kgqQazrBF9UH6UZ9ZryHaoUVLUmKv9ZiIf1EEDroUsFLvtCKbzCYZ0YX3d4nqvi8pehbla'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0a0b3-cc1e-7a83-bb08-54b5d1b079dc-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_7684140', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris:\n    1. Grand Hotel - 5 star, $350/night, spa, pool

## Fraction

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - return long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $100/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("fraction", 0.005),
            keep = ("fraction", 0.002),
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

## Token counter approx
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [11]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {'messages' : [HumanMessage(content = f'Find hotels in {city}')]},
        config = config
    )
    print(response)
    tokens = count_token(response['messages'])
    print(f'{city} : ~{tokens}, {len(response['messages'])} messages')
    print(f'{(response['messages'])}')


{'messages': [HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='ec450e2e-73d4-4647-b2d2-7ff8bee3e61b'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_7212329': 'El4KXAERTTIPxCACXUp/eDfJSS3b4nq5P+mC19klJryOFtH6t2KroeO7Os5MMC3Kb25GsacwyrdcldvZVDXikvDs/5fOX+ZgtuHvy6MQWAXD7Qm+tNiahdnnqOfPSjLq'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0a0b7-1c95-74a1-9273-893bdad46499-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_7212329', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris:\n    1. Grand Hotel - 5 star, $350/night, spa, pool

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id : str) -> str:
    """Mock functions to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [9]:
agent = create_agent(
    model = model,
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool" : {
                    "allowed_decisions" : ["approve", "edit", "reject"]
                },
                "read_email_tool" : False,
            }
        )
    ]
)

In [10]:
config = {"configurable" : {"thread_id" : "test-approve"}}

result = agent.invoke(
    {"messages" : [HumanMessage(content = "Send email to john@test.com with subject 'Hello' and body 'How are you ?'")]},
    config = config
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you ?'", additional_kwargs={}, response_metadata={}, id='0c33e031-477b-4a83-982f-d9e2a21bc532'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "body": "How are you ?", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_1937947': 'EmAKXgFpFH0TBJ/vqtSfom+b/6EM1y6BvJXpye1ssUg07j0UWdv/8fkHStgrlAt5BFJLcfLsp7jHN1ePf4kFcrfFTgW/VLRBdfWF9da6lKuHuKPlaM2kD74ufjwpSc4zRDY='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f372-3ff0-7352-ba4c-f2dc491eccf0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'body': 'How are you ?', 'recipient': 'john@test.com'}, 'id': 'call_1937947', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':

In [16]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving...")
    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config = config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: [{'type': 'text', 'text': "Email sent successfully to john@test.com with subject 'Hello'.", 'extras': {'signature': 'El4KXAERTTIPudDKBZPgy8F6if79H2W9zDrs+3xlHfnaySgKXSESApbvIqhonkzLqxRhekgW1BDyRf6ukKvk6k2RD631DJoI+I/MNri4gy54F9ZcV3DumhxUHntp+sxt'}}]


In [17]:
config = {"configurable" : {"thread_id" : "test-reject"}}

result = agent.invoke(
    {"messages" : [HumanMessage(content = "Send email to john@test.com with subject 'Hello' and body 'How are you ?'")]},
    config = config
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you ?'", additional_kwargs={}, response_metadata={}, id='098f9ff1-4a20-43dd-abfc-485a6a600778'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you ?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_8586360': 'El4KXAERTTIPIFniPMQDSR6u/C8xAs1m04nGazaJX4Lo+351Knn6geJYaBWflEEJO4MM+0wkfmdrcmbMjjBmIQh4aGinig8k1dlv9Ay6vZSl5tNWW9Ct8Mlyy7Xjd2hO'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0a15c-0e46-70c1-ae4c-0c2ca53da712-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you ?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_8586360', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143

In [18]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving...")
    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config = config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: [{'type': 'text', 'text': 'The email was not sent because the request was rejected. Let me know if you would like me to try again or need help with something else.', 'extras': {'signature': 'El4KXAERTTIP8CcOzwXXnbKLg0UB3ISysL2A1MdNHvvCRZWQR9TdYNX51GKIinDaZzcjbVhc43YkVQC1OJYNUCa2ELJM82cWV/qVXV6xgUv5ylYsLYkVuryGHfRQ1NeJ'}}]
